In [1]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Vikas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [58]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')
fake['label'] = 1
true['label'] = 0 
news= pd.concat([fake, true], ignore_index=True)
news.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1


In [59]:
print(f"Dataset shape: {news.shape}")
print("\nMissing values:\n", news.isnull().sum())

Dataset shape: (44898, 5)

Missing values:
 title      0
text       0
subject    0
date       0
label      0
dtype: int64


In [55]:
news.dropna(subset="text",axis=0)

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1
...,...,...,...,...,...
68511,"Donald Trump, Germany’s disfavored son – POLITICO","KALLSTADT, Germany — Few places in Germany are...",left-news,"Nov 14, 2017",0
68512,BREAKING: Hollywood Legend Just Died Of Terrib...,Hollywood loses yet another one of their deare...,left-news,"Nov 14, 2017",0
68513,Worst. President. Ever.,"As my 25th wedding anniversary approached, I t...",left-news,"Nov 14, 2017",0
68514,Don King drops N-word while introducing Donald...,Story highlights Trump was sitting in a chair ...,left-news,"Nov 14, 2017",0


In [9]:
news.title[0]

' Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing'

In [10]:
news.text[0]

'Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year. 2018 will be a great year for America!  Donald J. Trump (@realDonaldTrump) December 31, 2017Trump s tweet went down about as welll as you d expect.What kind of president sends a New Year s greeting like this despicable, petty, infantile gibberish? Only Trump! His lack of decency won t ev

In [11]:
news['content'] = news['title'] + ' ' + news['text']

In [12]:
news.content

0         Donald Trump Sends Out Embarrassing New Year’...
1         Drunk Bragging Trump Staffer Started Russian ...
2         Sheriff David Clarke Becomes An Internet Joke...
3         Trump Is So Obsessed He Even Has Obama’s Name...
4         Pope Francis Just Called Out Donald Trump Dur...
                               ...                        
44893    'Fully committed' NATO backs new U.S. approach...
44894    LexisNexis withdrew two products from Chinese ...
44895    Minsk cultural hub becomes haven from authorit...
44896    Vatican upbeat on possibility of Pope Francis ...
44897    Indonesia to buy $1.14 billion worth of Russia...
Name: content, Length: 44898, dtype: object

In [14]:
ps = PorterStemmer()
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [ps.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content
news['content'] = news['content'].apply(stemming)
print("Sample processed content:\n", news['content'].iloc[2])

Sample processed content:
 sheriff david clark becom internet joke threaten poke peopl eye friday reveal former milwauke sheriff david clark consid homeland secur secretari donald trump administr email scandal januari brief run plane clark fellow passeng dan black later detain polic reason whatsoev except mayb feel hurt clark messag polic stop black deplan search warrant execut fbi see exchang clark call fake news even though copi search warrant internet unintimid lib media attempt smear discredit fake news report design silenc former sheriff tweet continu poke eye sharp stick bitch slap scum bag til get attack better peopl maga unintimid lib media attempt smear discredit fake news report design silenc continu poke eye sharp stick bitch slap scum bag til get attack better peopl maga pic twitter com xtzw pdu b david clark jr sheriffclark decemb stop break news lie lib media make fake news smear antidot go right punch nose make tast blood noth get bulli like lie lib media attent better g

In [15]:
X = news['content'].values
y = news['label'].values

vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

print(f"Shape of TF-IDF matrix: {X.shape}")

Shape of TF-IDF matrix: (44898, 89868)


In [16]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=2)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (35918, 89868)
Testing set shape: (8980, 89868)


In [17]:
# Initialize and train model
model = LogisticRegression()
model.fit(X_train, Y_train)

LogisticRegression()

In [18]:
train_y_pred = model.predict(X_train)
train_accuracy = accuracy_score(train_y_pred, Y_train)
print(f'Training Accuracy: {train_accuracy:.4f}')

test_y_pred = model.predict(X_test)
test_accuracy = accuracy_score(test_y_pred, Y_test)
print(f'Testing Accuracy: {test_accuracy:.4f}')

Training Accuracy: 0.9909
Testing Accuracy: 0.9882


In [10]:
_, test_indices = train_test_split(news_df.index, test_size=0.2, stratify=news_df['label'], random_state=2)

original_index = test_indices[10]

original_data = news_df.loc[original_index]

print("Original Index in Dataset:", original_index)
print("\nTitle:", original_data['title'])
print("\nText:", original_data['text'])
print("\nProcessed Content:", original_data['content'])
print("\nTrue Label:", "Fake" if original_data['label'] == 1 else "Real")

input_data = X_test[10]
prediction = model.predict(input_data)

print("\nPredicted Label:", "The News is Fake" if prediction[0] == 1 else "The News is Real")

Original Index in Dataset: 18576

Title: MANCHESTER: MUSLIM WOMAN Wearing HAND GRENADE, Machine Gun, Knife Images On Burqa Says She’s Worried About Facing “Islamaphobia” After Terror Attack [VIDEO]

Text: A Muslim woman was criticised after she wore a burka emblazoned with weapons in a TV interview about the Manchester terror attackThe woman, named only as  Sid , was part of a group of Muslims interviewed by Channel 4 News journalist Krishnan Guru-Murthy in the Rusholme area of Manchester, which is home to the largest Libyan community in Europe.The group vehemently condemned the Manchester Arena attack by 22-year-old suicide bomber Salman Abedi, who is of Libyan descent.But some viewers criticised Sid over her decision to wear a burka printed with a pistol, grenade, knife and machine gun spelling the word  love .One wrote:  Channel 4 news contributor in full hijab, top to toe with comedy glasses, images of knife & hand grenade emblazoned across the front! Ffs.The interview with the Mus